In [1]:
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
!pip install pyspark -q

In [6]:
#zadanie 1
spark = SparkSession.builder \
    .appName("SparkSQL_Lab10") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# tworzenie przykładowe dane sprzedażowe
data = {
    "id": range(1, 101),
    "region": ["North", "South", "East", "West"] * 25,
    "product": ["Laptop", "Phone", "Tablet", "Monitor"] * 25,
    "amount": [round(500 + i * 13.5, 2) for i in range(100)],
    "quantity": [i % 10 + 1 for i in range(100)]
}

df_pandas = pd.DataFrame(data)
df_pandas.to_parquet("sales.parquet", index=False)
print("Plik sales.parquet zapisany.")

df = spark.read.parquet("sales.parquet")
print("=== Pierwsze wiersze ===")
df.show(10)

print("=== Schemat danych ===")
df.printSchema()

Plik sales.parquet zapisany.
=== Pierwsze wiersze ===
+---+------+-------+------+--------+
| id|region|product|amount|quantity|
+---+------+-------+------+--------+
|  1| North| Laptop| 500.0|       1|
|  2| South|  Phone| 513.5|       2|
|  3|  East| Tablet| 527.0|       3|
|  4|  West|Monitor| 540.5|       4|
|  5| North| Laptop| 554.0|       5|
|  6| South|  Phone| 567.5|       6|
|  7|  East| Tablet| 581.0|       7|
|  8|  West|Monitor| 594.5|       8|
|  9| North| Laptop| 608.0|       9|
| 10| South|  Phone| 621.5|      10|
+---+------+-------+------+--------+
only showing top 10 rows
=== Schemat danych ===
root
 |-- id: long (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: long (nullable = true)



In [8]:
#zadanie 2
customers_data = {
    "customer_id": range(1, 26),
    "name": [f"Customer_{i}" for i in range(1, 26)],
    "region": ["North", "South", "East", "West", "North"] * 5,
    "age": [20 + i for i in range(25)]
}

pd.DataFrame(customers_data).to_csv("customers.csv", index=False)
print("Plik customers.csv zapisany.")

df_csv = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

print("\n=== Dane CSV ===")
df_csv.show(10)
df_csv.printSchema()

df_csv.createOrReplaceTempView("customers")

# Sprawdzenie zapytaniem SQL
result = spark.sql("SELECT * FROM customers LIMIT 10")
result.show()

df.createOrReplaceTempView("sales")

result2 = spark.sql("SELECT * FROM sales LIMIT 10")
result2.show()

Plik customers.csv zapisany.

=== Dane CSV ===
+-----------+-----------+------+---+
|customer_id|       name|region|age|
+-----------+-----------+------+---+
|          1| Customer_1| North| 20|
|          2| Customer_2| South| 21|
|          3| Customer_3|  East| 22|
|          4| Customer_4|  West| 23|
|          5| Customer_5| North| 24|
|          6| Customer_6| North| 25|
|          7| Customer_7| South| 26|
|          8| Customer_8|  East| 27|
|          9| Customer_9|  West| 28|
|         10|Customer_10| North| 29|
+-----------+-----------+------+---+
only showing top 10 rows
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- age: integer (nullable = true)

+-----------+-----------+------+---+
|customer_id|       name|region|age|
+-----------+-----------+------+---+
|          1| Customer_1| North| 20|
|          2| Customer_2| South| 21|
|          3| Customer_3|  East| 22|
|          4| Customer_4|  W

In [9]:
#zadanie 3
query_agg = """
SELECT
    product,
    COUNT(*)        AS liczba_transakcji,
    SUM(amount)     AS suma_sprzedazy,
    AVG(amount)     AS srednia_kwota,
    MAX(amount)     AS max_kwota,
    MIN(amount)     AS min_kwota
FROM sales
GROUP BY product
ORDER BY suma_sprzedazy DESC
"""

df_agg = spark.sql(query_agg)
print("=== Agregacje po produkcie ===")
df_agg.show()

query_group = """
SELECT
    region,
    product,
    COUNT(*)    AS liczba,
    SUM(amount) AS przychod
FROM sales
GROUP BY region, product
ORDER BY region, przychod DESC
"""

df_group = spark.sql(query_group)
print("\n=== Grupowanie po regionie i produkcie ===")
df_group.show(20)

query_filter = """
SELECT *
FROM sales
WHERE amount > 1000
  AND region = 'North'
ORDER BY amount DESC
"""

df_filter = spark.sql(query_filter)
print("\n=== Transakcje powyżej 1000 w regionie North ===")
df_filter.show()

query_join = """
SELECT
    s.id,
    s.product,
    s.amount,
    s.region,
    c.name      AS klient,
    c.age       AS wiek_klienta
FROM sales s
JOIN customers c
    ON s.region = c.region
WHERE s.amount > 800
ORDER BY s.amount DESC
"""

df_join = spark.sql(query_join)
print("\n=== JOIN sprzedaży z klientami ===")
df_join.show(15)

df_agg.write.mode("overwrite").parquet("output/wyniki_agregacje.parquet")
print("\nZapisano do output/wyniki_agregacje.parquet")

df_join.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("output/wyniki_join.csv")
print("\nZapisano do output/wyniki_join.csv")

=== Agregacje po produkcie ===
+-------+-----------------+--------------+-------------+---------+---------+
|product|liczba_transakcji|suma_sprzedazy|srednia_kwota|max_kwota|min_kwota|
+-------+-----------------+--------------+-------------+---------+---------+
|Monitor|               25|       29712.5|       1188.5|   1836.5|    540.5|
| Tablet|               25|       29375.0|       1175.0|   1823.0|    527.0|
|  Phone|               25|       29037.5|       1161.5|   1809.5|    513.5|
| Laptop|               25|       28700.0|       1148.0|   1796.0|    500.0|
+-------+-----------------+--------------+-------------+---------+---------+


=== Grupowanie po regionie i produkcie ===
+------+-------+------+--------+
|region|product|liczba|przychod|
+------+-------+------+--------+
|  East| Tablet|    25| 29375.0|
| North| Laptop|    25| 28700.0|
| South|  Phone|    25| 29037.5|
|  West|Monitor|    25| 29712.5|
+------+-------+------+--------+


=== Transakcje powyżej 1000 w regionie Nor

In [10]:
spark.stop()